In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

RAW_DATA_PATH = "/content/drive/MyDrive/Meridian_Churn_Capstone/Data/Raw"

customers = pd.read_csv(os.path.join(RAW_DATA_PATH, "customers.csv"))
products = pd.read_csv(os.path.join(RAW_DATA_PATH, "products.csv"))
orders = pd.read_csv(os.path.join(RAW_DATA_PATH, "orders.csv"))
order_items = pd.read_csv(os.path.join(RAW_DATA_PATH, "order_items.csv"))
support_tickets = pd.read_csv(os.path.join(RAW_DATA_PATH, "support_tickets.csv"))

# Preserve the original data in memory
raw_datasets = {
    "customers": customers.copy(deep=True),
    "products": products.copy(deep=True),
    "orders": orders.copy(deep=True),
    "order_items": order_items.copy(deep=True),
    "support_tickets": support_tickets.copy(deep=True)
 }

# Create separate working copies for cleaning
customers_clean = customers.copy(deep=True)
products_clean = products.copy(deep=True)
orders_clean = orders.copy(deep=True)
order_items_clean = order_items.copy(deep=True)
support_tickets_clean = support_tickets.copy(deep=True)

print("All five datasets loaded successfully!")

All five datasets loaded successfully!


In [3]:
for name, df in raw_datasets.items():
    print(f"{name}: {df.shape[0]:,} rows and {df.shape[1]} columns")

customers: 6,090 rows and 14 columns
products: 220 rows and 9 columns
orders: 28,297 rows and 9 columns
order_items: 73,505 rows and 5 columns
support_tickets: 5,200 rows and 8 columns


In [4]:
# STEP 6.2A: Inspect exact duplicate rows in all five datasets

for name, df in raw_datasets.items():
    exact_duplicates = df.duplicated().sum()

    print(f"\n{name.upper()}")
    print(f"Total rows: {len(df):,}")
    print(f"Exact duplicate rows: {exact_duplicates:,}")

    if exact_duplicates > 0:
        print("Sample duplicate records:")
        display(df[df.duplicated(keep=False)].head(10))
    else:
        print("No exact duplicate rows found.")


CUSTOMERS
Total rows: 6,090
Exact duplicate rows: 0
No exact duplicate rows found.

PRODUCTS
Total rows: 220
Exact duplicate rows: 0
No exact duplicate rows found.

ORDERS
Total rows: 28,297
Exact duplicate rows: 25
Sample duplicate records:


,order_id,customer_id,order_date,channel,order_status,payment_method,shipping_cost,discount_pct,total_amount
2873,ORD102482,CUST005630,2025-03-17,Mobile App,Completed,Credit Card,0.00,0,2852.97
3287,ORD120778,CUST003051,2026-01-25,In-Store,Completed,Store Credit,0.00,0,1264.87
4065,ORD118529,CUST004560,2025-01-08,In-Store,Pending,Credit Card,0.00,0,1106.52
4740,ORD126857,CUST000901,2026-03-01,Mobile App,Completed,Store Credit,12.99,15,2094.27
4799,ORD122218,CUST004944,2026-03-16,Online,Completed,Credit Card,0.00,0,1498.98
4934,ORD125341,CUST002550,2025-07-10,In-Store,Completed,PayPal,0.00,0,1674.84
5349,ORD126857,CUST000901,2026-03-01,Mobile App,Completed,Store Credit,12.99,15,2094.27
5431,ORD127030,CUST005262,2025-02-16,Mobile App,Completed,Credit Card,7.99,10,563.00
5795,ORD103787,CUST001807,2026-02-06,Mobile App,Completed,Debit Card,0.00,0,2633.64
5870,ORD104809,CUST005324,2025-08-05,Mobile App,Completed,Gift Card,12.99,5,549.19



ORDER_ITEMS
Total rows: 73,505
Exact duplicate rows: 40
Sample duplicate records:


,order_item_id,order_id,product_id,quantity,unit_price_at_purchase
78,OI539793,ORD113908,PROD1211,1,78.61
834,OI566718,ORD104885,PROD1186,1,91.94
1103,OI508123,ORD102602,PROD1125,1,223.63
2717,OI536778,ORD113389,PROD1054,3,130.22
4064,OI520895,ORD113530,PROD1138,1,724.57
10147,OI557481,ORD118899,PROD1037,2,663.90
11400,OI520895,ORD113530,PROD1138,1,724.57
13910,OI514791,ORD109726,PROD1181,3,383.82
14998,OI514791,ORD109726,PROD1181,3,383.82
16315,OI563770,ORD125553,PROD1213,1,448.11



SUPPORT_TICKETS
Total rows: 5,200
Exact duplicate rows: 0
No exact duplicate rows found.


In [5]:
# STEP 6.2B: Inspect repeated primary-key values

primary_keys = {
    "customers": ("customers", "customer_id"),
    "products": ("products", "product_id"),
    "orders": ("orders", "order_id"),
    "order_items": ("order_items", "order_item_id"),
    "support_tickets": ("support_tickets", "ticket_id")
}

for label, (dataset_name, key) in primary_keys.items():
    df = raw_datasets[dataset_name]

    duplicate_mask = df[key].duplicated(keep=False)

    print(f"\n{label.upper()} — {key}")
    print(f"Rows with repeated IDs: {duplicate_mask.sum():,}")
    print(f"Distinct repeated IDs: {df.loc[duplicate_mask, key].nunique():,}")

    if duplicate_mask.any():
          display(
             df.loc[duplicate_mask]
               .sort_values(key)
               .head(10)
        )


CUSTOMERS — customer_id
Rows with repeated IDs: 0
Distinct repeated IDs: 0

PRODUCTS — product_id
Rows with repeated IDs: 0
Distinct repeated IDs: 0

ORDERS — order_id
Rows with repeated IDs: 50
Distinct repeated IDs: 25


,order_id,customer_id,order_date,channel,order_status,payment_method,shipping_cost,discount_pct,total_amount
2873,ORD102482,CUST005630,2025-03-17,Mobile App,Completed,Credit Card,0.00,0,2852.97
10265,ORD102482,CUST005630,2025-03-17,Mobile App,Completed,Credit Card,0.00,0,2852.97
19831,ORD103359,CUST005402,2025-05-29,In-Store,Completed,Credit Card,0.00,0,420.05
15944,ORD103359,CUST005402,2025-05-29,In-Store,Completed,Credit Card,0.00,0,420.05
5795,ORD103787,CUST001807,2026-02-06,Mobile App,Completed,Debit Card,0.00,0,2633.64
23352,ORD103787,CUST001807,2026-02-06,Mobile App,Completed,Debit Card,0.00,0,2633.64
20404,ORD103914,CUST005431,2024-12-12,In-Store,Completed,Debit Card,5.99,5,2571.00
17094,ORD103914,CUST005431,2024-12-12,In-Store,Completed,Debit Card,5.99,5,2571.00
14105,ORD104413,CUST004015,2024-09-11,Mobile App,Completed,Store Credit,4.99,5,206.07
19106,ORD104413,CUST004015,2024-09-11,Mobile App,Completed,Store Credit,4.99,5,206.07



ORDER_ITEMS — order_item_id
Rows with repeated IDs: 80
Distinct repeated IDs: 40


,order_item_id,order_id,product_id,quantity,unit_price_at_purchase
41701,OI501045,ORD105556,PROD1058,2,503.85
46542,OI501045,ORD105556,PROD1058,2,503.85
17274,OI502291,ORD100604,PROD1049,2,20.33
51376,OI502291,ORD100604,PROD1049,2,20.33
40523,OI502812,ORD102369,PROD1213,2,457.96
16457,OI502812,ORD102369,PROD1213,2,457.96
71916,OI503791,ORD122023,PROD1071,2,338.19
72663,OI503791,ORD122023,PROD1071,2,338.19
53434,OI504321,ORD115140,PROD1175,1,514.16
55153,OI504321,ORD115140,PROD1175,1,514.16



SUPPORT_TICKETS — ticket_id
Rows with repeated IDs: 0
Distinct repeated IDs: 0


In [6]:
# STEP 6.2C: Identify possible duplicate customers

print("Rows with repeated email addresses, ignoring case:")

email_normalized = (
    customers_clean["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

duplicate_email_mask = (
    email_normalized.notna()
    & email_normalized.duplicated(keep=False)
)

display(
    customers_clean.loc[
        duplicate_email_mask,
        ["customer_id", "first_name", "last_name", "email", "signup_date"]
    ].assign(
        email_normalized=email_normalized[duplicate_email_mask]
     ).sort_values("email_normalized").head(30)
)

print(
    "\nRows with repeated normalized emails:",
    int(duplicate_email_mask.sum())
)

Rows with repeated email addresses, ignoring case:


,customer_id,first_name,last_name,email,signup_date,email_normalized
798,CUST006076,Amanda,Hudson,AMANDA.HUDSON535@HOTMAIL.COM,2022-06-10,amanda.hudson535@hotmail.com
5266,CUST003328,Amanda,Hudson,amanda.hudson535@hotmail.com,2022-06-10,amanda.hudson535@hotmail.com
1246,CUST000208,Amy,Powell,amy.powell739@hotmail.com,2025-10-25,amy.powell739@hotmail.com
754,CUST006050,Amy,Powell,AMY.POWELL739@HOTMAIL.COM,2025-10-25,amy.powell739@hotmail.com
2585,CUST000722,Angela,Wheeler,angela.wheeler652@outlook.com,2021-04-20,angela.wheeler652@outlook.com
5809,CUST006042,Angela,Wheeler,ANGELA.WHEELER652@OUTLOOK.COM,2021-04-20,angela.wheeler652@outlook.com
2543,CUST003179,Angela,Wood,angela.wood172@hotmail.com,2023-11-03,angela.wood172@hotmail.com
1550,CUST006024,Angela,Wood,ANGELA.WOOD172@HOTMAIL.COM,2023-11-03,angela.wood172@hotmail.com
138,CUST003180,Anna,Nelson,anna.nelson34@yahoo.com,2025-09-02,anna.nelson34@yahoo.com
4117,CUST006054,Anna,Nelson,ANNA.NELSON34@YAHOO.COM,2025-09-02,anna.nelson34@yahoo.com



Rows with repeated normalized emails: 176


In [7]:
# STEP 6.2D: Verify whether repeated transaction IDs have identical records

for dataset_name, key in [
    ("orders", "order_id"),
    ("order_items", "order_item_id")
]:
    df = raw_datasets[dataset_name]

    repeated = df[df[key].duplicated(keep=False)].copy()

    # Count distinct versions of each repeated ID
    versions = repeated.groupby(key).apply(
        lambda group: len(group.drop_duplicates()),
        include_groups=False
    )

    identical_ids = (versions == 1).sum()
    conflicting_ids = (versions > 1).sum()

    print(f"\n{dataset_name.upper()}")
    print(f"Repeated IDs examined: {len(versions)}")
    print(f"IDs whose rows are identical: {identical_ids}")
    print(f"IDs with conflicting records: {conflicting_ids}")

    if conflicting_ids > 0:
        conflicting_keys = versions[versions > 1].index
        display(
            repeated[repeated[key].isin(conflicting_keys)]
            .sort_values(key)
        )


ORDERS
Repeated IDs examined: 25
IDs whose rows are identical: 25
IDs with conflicting records: 0

ORDER_ITEMS
Repeated IDs examined: 40
IDs whose rows are identical: 40
IDs with conflicting records: 0


In [8]:
# STEP 6.2E: Summarize possible duplicate customer profiles

customer_check = customers_clean.copy()

customer_check["email_normalized"] = (
    customer_check["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

customer_check["name_normalized"] = (
    customer_check["first_name"].fillna("").astype(str).str.strip().str.lower()
    + " "
    + customer_check["last_name"].fillna("").astype(str).str.strip().str.lower()
).str.strip()

repeated_emails = customer_check[
    customer_check["email_normalized"].notna()
    & customer_check["email_normalized"].duplicated(keep=False)
]

email_summary = (
    repeated_emails.groupby("email_normalized")
    .agg(
        number_of_records=("customer_id", "size"),
        distinct_customer_ids=("customer_id", "nunique"),
        distinct_names=("name_normalized", "nunique"),
        distinct_signup_dates=("signup_date", "nunique")
    )
    .reset_index()
)

print("Distinct repeated email addresses:", len(email_summary))
print("\nSummary of repeated email groups:")
display(email_summary.head(20))

print("\nGroups with different names or signup dates:")
display(
    email_summary[
        (email_summary["distinct_names"] > 1)
        | (email_summary["distinct_signup_dates"] > 1)
    ].head(20)
)

Distinct repeated email addresses: 88

Summary of repeated email groups:


,email_normalized,number_of_records,distinct_customer_ids,distinct_names,distinct_signup_dates
0,amanda.hudson535@hotmail.com,2,2,1,1
1,amy.powell739@hotmail.com,2,2,1,1
2,angela.wheeler652@outlook.com,2,2,1,1
3,angela.wood172@hotmail.com,2,2,1,1
4,anna.nelson34@yahoo.com,2,2,1,1
5,anthony.elliott607@gmail.com,2,2,1,1
6,anthony.knox154@yahoo.com,2,2,1,1
7,benjamin.archer384@outlook.com,2,2,1,1
8,bridget.espinoza132@gmail.com,2,2,1,2
9,bryan.blackwell437@hotmail.com,2,2,1,1



Groups with different names or signup dates:


,email_normalized,number_of_records,distinct_customer_ids,distinct_names,distinct_signup_dates
8,bridget.espinoza132@gmail.com,2,2,1,2
17,crystal.dixon494@gmail.com,2,2,1,2
30,gary.webb64@gmail.com,2,2,1,2
31,glenn.jackson458@yahoo.com,2,2,1,2
40,jeffrey.graham489@yahoo.com,2,2,1,2
41,jeffrey.hill919@gmail.com,2,2,1,2
48,justin.hartman473@yahoo.com,2,2,1,2
50,kim.sparks326@outlook.com,2,2,1,2
57,matthew.wall281@gmail.com,2,2,1,2
73,richard.sanford565@yahoo.com,2,2,1,2


In [9]:
# STEP 6.2F: Remove verified exact duplicate transaction rows
# The original raw_datasets remain unchanged.

orders_before = len(orders_clean)
items_before = len(order_items_clean)

orders_clean = orders_clean.drop_duplicates().copy()
order_items_clean = order_items_clean.drop_duplicates().copy()

print("ORDERS")
print("Rows before:", f"{orders_before:,}")
print("Rows after:", f"{len(orders_clean):,}")
print("Rows removed:", f"{orders_before - len(orders_clean):,}")

print("\nORDER ITEMS")
print("Rows before:", f"{items_before:,}")
print("Rows after:", f"{len(order_items_clean):,}")
print("Rows removed:", f"{items_before - len(order_items_clean):,}")


ORDERS
Rows before: 28,297
Rows after: 28,272
Rows removed: 25

ORDER ITEMS
Rows before: 73,505
Rows after: 73,465
Rows removed: 40


In [10]:
# STEP 6.3A: Remove accidental whitespace and inspect column data types

clean_datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "support_tickets": support_tickets_clean
}

for name, df in clean_datasets.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

    print(f"\n{name.upper()}")
    display(
        pd.DataFrame({
            "column": df.columns,
            "data_type": df.dtypes.astype(str).values,
            "missing_values": df.isna().sum().values
        })
    )


CUSTOMERS


,column,data_type,missing_values
0,customer_id,string,0
1,first_name,string,0
2,last_name,string,0
3,email,string,242
4,phone,string,423
5,gender,string,0
6,date_of_birth,string,181
7,signup_date,string,0
8,state,string,0
9,city,string,121



PRODUCTS


,column,data_type,missing_values
0,product_id,string,0
1,product_name,string,0
2,category,string,0
3,sub_category,string,11
4,brand,string,0
5,unit_cost,float64,0
6,unit_price,string,0
7,launch_date,string,0
8,is_active,string,0



ORDERS


,column,data_type,missing_values
0,order_id,string,0
1,customer_id,string,0
2,order_date,string,424
3,channel,string,0
4,order_status,string,0
5,payment_method,string,0
6,shipping_cost,float64,0
7,discount_pct,int64,0
8,total_amount,float64,0



ORDER_ITEMS


,column,data_type,missing_values
0,order_item_id,string,0
1,order_id,string,0
2,product_id,string,0
3,quantity,int64,0
4,unit_price_at_purchase,float64,1469



SUPPORT_TICKETS


,column,data_type,missing_values
0,ticket_id,string,0
1,customer_id,string,0
2,ticket_date,string,0
3,issue_category,string,0
4,channel,string,0
5,resolution_time_hours,float64,0
6,satisfaction_score,float64,592
7,ticket_text,string,0


In [11]:
# STEP 6.3B: Convert date columns to datetime

date_columns = {
    "customers": ["date_of_birth", "signup_date"],
    "products": ["launch_date"],
    "orders": ["order_date"],
    "order_items": [],
    "support_tickets": ["ticket_date"]
}

for name, columns in date_columns.items():
    df = clean_datasets[name]

    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            format="mixed",
            errors="coerce"
        )

        print(
            f"{name}.{col}: "
            f"{df[col].isna().sum():,} missing or unparseable values" )

customers.date_of_birth: 181 missing or unparseable values
customers.signup_date: 0 missing or unparseable values
products.launch_date: 0 missing or unparseable values
orders.order_date: 424 missing or unparseable values
support_tickets.ticket_date: 0 missing or unparseable values


In [12]:
# STEP 6.3C: Convert numeric fields to numeric data types

numeric_columns = {
    "products": ["unit_cost", "unit_price"],
    "orders": ["shipping_cost", "discount_pct", "total_amount"],
    "order_items": ["quantity", "unit_price_at_purchase"],
    "support_tickets": [
        "resolution_time_hours",
        "satisfaction_score"
    ]
}

for name, columns in numeric_columns.items():
    df = clean_datasets[name]

    for col in columns:
        # Remove currency symbols, commas, and surrounding spaces.
        # This handles values such as "$1,250.50".
        cleaned_text = (
            df[col]
            .astype("string")
            .str.replace(r"[$,]", "", regex=True)
            .str.strip()
        )

        df[col] = pd.to_numeric(cleaned_text, errors="coerce")

        print(
            f"{name}.{col}: "
            f"{df[col].isna().sum():,} missing or unparseable values"
        )

products.unit_cost: 0 missing or unparseable values
products.unit_price: 0 missing or unparseable values
orders.shipping_cost: 0 missing or unparseable values
orders.discount_pct: 0 missing or unparseable values
orders.total_amount: 0 missing or unparseable values
order_items.quantity: 0 missing or unparseable values
order_items.unit_price_at_purchase: 1,469 missing or unparseable values
support_tickets.resolution_time_hours: 0 missing or unparseable values
support_tickets.satisfaction_score: 592 missing or unparseable values


In [13]:
# STEP 6.3D: Verify converted data types

for name, df in clean_datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes.to_string())


CUSTOMERS
customer_id          string[python]
first_name           string[python]
last_name            string[python]
email                string[python]
phone                string[python]
gender               string[python]
date_of_birth        datetime64[ns]
signup_date          datetime64[ns]
state                string[python]
city                 string[python]
membership_tier      string[python]
preferred_channel    string[python]
marketing_opt_in     string[python]
referral_source      string[python]

PRODUCTS
product_id      string[python]
product_name    string[python]
category        string[python]
sub_category    string[python]
brand           string[python]
unit_cost              Float64
unit_price             Float64
launch_date     datetime64[ns]
is_active       string[python]

ORDERS
order_id          string[python]
customer_id       string[python]
order_date        datetime64[ns]
channel           string[python]
order_status      string[python]
payment_method    strin

In [14]:
# STEP 6.4: Standardize categorical values

# Standardize capitalization in selected categorical columns.
title_case_columns = {
    "customers": [
        "gender", "state", "city", "membership_tier",
        "preferred_channel", "referral_source"
    ],
    "products": ["category", "sub_category", "brand"],
    "orders": ["channel", "order_status", "payment_method"],
    "support_tickets": ["issue_category", "channel"]
}

for name, columns in title_case_columns.items():
    df = clean_datasets[name]

    for col in columns:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .str.title()
            )

# Standardize marketing consent values.
customers_clean["marketing_opt_in"] = (
    customers_clean["marketing_opt_in"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map({
        "Y": "Yes",
        "YES": "Yes",
        "1": "Yes",
        "TRUE": "Yes",
        "N": "No",
        "NO": "No",
        "0": "No",
        "FALSE": "No"
    })
)

# Standardize product active-status values.
products_clean["is_active"] = (
    products_clean["is_active"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map({
        "Y": "Yes",
        "YES": "Yes",
        "1": "Yes",
        "TRUE": "Yes",
        "N": "No",
        "NO": "No",
        "0": "No",
        "FALSE": "No"
    })
)

print("Categorical standardization completed.")



Categorical standardization completed.


In [15]:
# STEP 6.4B: Inspect the standardized categories

for name, columns in {
    "customers": [
        "gender", "membership_tier", "preferred_channel",
        "marketing_opt_in"
    ],
    "products": ["category", "is_active"],
    "orders": ["channel", "order_status", "payment_method"],
    "support_tickets": ["issue_category", "channel"]
}.items():
    print(f"\n{name.upper()}")

    for col in columns:
        print(f"\n{col}:")
        print(clean_datasets[name][col].value_counts(dropna=False).to_string())


CUSTOMERS

gender:
gender
M        2530
F        2363
Other    1197

membership_tier:
membership_tier
Bronze      2785
Silver      1768
Gold        1126
Platinum     411

preferred_channel:
preferred_channel
Mobile App    2053
In-Store      2041
Online        1996

marketing_opt_in:
marketing_opt_in
Yes    2453
No     2446
NaN    1191

PRODUCTS

category:
category
Electronics               48
Beauty & Personal Care    45
Home & Kitchen            45
Sports & Outdoors         44
Apparel                   38

is_active:
is_active
Yes    194
No      26

ORDERS

channel:
channel
Online        9535
In-Store      9486
Mobile App    9251

order_status:
order_status
Completed    25481
Returned      1230
Cancelled      976
Pending        585

payment_method:
payment_method
Gift Card       5761
Paypal          5731
Debit Card      5639
Credit Card     5630
Store Credit    5511

SUPPORT_TICKETS

issue_category:
issue_category
Product Question      715
Refund Status         671
Damaged Item      

In [16]:
# STEP 6.5A: Report missing values in all five cleaned datasets

missing_reports = []

for name, df in clean_datasets.items():
    for col in df.columns:
        missing_count = int(df[col].isna().sum())

        missing_reports.append({
            "dataset": name,
            "column": col,
            "missing_count": missing_count,
            "missing_percent": round(
                (missing_count / len(df)) * 100, 2
            ) if len(df) else 0
        })

missing_report = pd.DataFrame(missing_reports)

missing_report = (
    missing_report[missing_report["missing_count"] > 0]
    .sort_values(
        ["dataset", "missing_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(missing_report)

,dataset,column,missing_count,missing_percent
0,customers,referral_source,1705,28.00
1,customers,marketing_opt_in,1191,19.56
2,customers,phone,423,6.95
3,customers,email,242,3.97
4,customers,date_of_birth,181,2.97
5,customers,city,121,1.99
6,order_items,unit_price_at_purchase,1469,2.00
7,orders,order_date,424,1.50
8,products,sub_category,11,5.00
9,support_tickets,satisfaction_score,592,11.38


In [17]:
# STEP 6.5B: Create selected missingness indicators

missing_flag_columns = {
    "customers": [
        "email",
        "phone",
        "date_of_birth",
        "city",
        "marketing_opt_in",
        "referral_source"
    ],
    "products": ["sub_category"],
    "orders": ["order_date"],
    "order_items": ["unit_price_at_purchase"],
    "support_tickets": ["satisfaction_score"]
}

for dataset_name, columns in missing_flag_columns.items():
    df = clean_datasets[dataset_name]

    for col in columns:
        flag_name = f"{col}_missing"

        df[flag_name] = df[col].isna().astype("int8")

print("Missingness flags created.")

for name, df in clean_datasets.items():
    flags = [col for col in df.columns if col.endswith("_missing")]

    if flags:
        print(f"\n{name.upper()}")
        display(df[flags].sum().rename("missing_flag_count").to_frame())

Missingness flags created.

CUSTOMERS


,missing_flag_count
email_missing,242
phone_missing,423
date_of_birth_missing,181
city_missing,121
marketing_opt_in_missing,1191
referral_source_missing,1705



PRODUCTS


,missing_flag_count
sub_category_missing,11



ORDERS


,missing_flag_count
order_date_missing,424



ORDER_ITEMS


,missing_flag_count
unit_price_at_purchase_missing,1469



SUPPORT_TICKETS


,missing_flag_count
satisfaction_score_missing,592


In [18]:
# STEP 6.6A: Inspect potentially invalid values

print("CUSTOMERS")

print(
    "Birth dates before 1900:",
    int((customers_clean["date_of_birth"] < "1900-01-01").sum())
)

print(
    "Birth dates after the snapshot date:",
    int((customers_clean["date_of_birth"] > "2026-07-01").sum())
)

print(
    "Signup dates after the snapshot date:",
    int((customers_clean["signup_date"] > "2026-07-01").sum())
)

print("\nPRODUCTS")

print(
    "Products where unit cost exceeds unit price:",
    int((products_clean["unit_cost"] > products_clean["unit_price"]).sum())
)

print("\nORDERS")

print(
    "Negative shipping costs:",
    int((orders_clean["shipping_cost"] < 0).sum())
)

print(
    "Discounts outside 0–25:",
    int(
        (
            (orders_clean["discount_pct"] < 0)
            | (orders_clean["discount_pct"] > 25)
        ).sum()
    )
)

print("\nORDER ITEMS")

print(
    "Zero or negative quantities:",
    int((order_items_clean["quantity"] <= 0).sum())
)

print(
    "Negative purchase prices:",
    int((order_items_clean["unit_price_at_purchase"] < 0).sum())
)

print("\nSUPPORT TICKETS")

print(
    "Negative resolution times:",
    int((support_tickets_clean["resolution_time_hours"] < 0).sum())
)

print(
    "Satisfaction scores outside 1–5:",
    int(
        (
            (support_tickets_clean["satisfaction_score"] < 1)
            | (support_tickets_clean["satisfaction_score"] > 5)
        ).sum()
    )
)

CUSTOMERS
Birth dates before 1900: 8
Birth dates after the snapshot date: 7
Signup dates after the snapshot date: 0

PRODUCTS
Products where unit cost exceeds unit price: 6

ORDERS
Negative shipping costs: 8
Discounts outside 0–25: 0

ORDER ITEMS
Zero or negative quantities: 18
Negative purchase prices: 0

SUPPORT TICKETS
Negative resolution times: 6
Satisfaction scores outside 1–5: 0


In [19]:
missing_reports = []

for name, df in clean_datasets.items():
    for col in df.columns:
        missing_count = int(df[col].isna().sum())

        missing_reports.append({
            "dataset": name,
            "column": col,
            "missing_count": missing_count,
            "missing_percent": round(
                (missing_count / len(df)) * 100, 2
            ) if len(df) else 0
        })

missing_report = pd.DataFrame(missing_reports)

missing_report = (
    missing_report[missing_report["missing_count"] > 0]
    .sort_values(
        ["dataset", "missing_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(missing_report)

,dataset,column,missing_count,missing_percent
0,customers,referral_source,1705,28.00
1,customers,marketing_opt_in,1191,19.56
2,customers,phone,423,6.95
3,customers,email,242,3.97
4,customers,date_of_birth,181,2.97
5,customers,city,121,1.99
6,order_items,unit_price_at_purchase,1469,2.00
7,orders,order_date,424,1.50
8,products,sub_category,11,5.00
9,support_tickets,satisfaction_score,592,11.38


In [20]:
missing_flag_columns = {
    "customers": [
        "email", "phone", "date_of_birth", "city",
        "marketing_opt_in", "referral_source"
    ],
    "products": ["sub_category"],
    "orders": ["order_date"],
    "order_items": ["unit_price_at_purchase"],
    "support_tickets": ["satisfaction_score"]
}

for dataset_name, columns in missing_flag_columns.items():
    df = clean_datasets[dataset_name]

    for col in columns:
        flag_name = f"{col}_missing"
        df[flag_name] = df[col].isna().astype("int8")

print("Missingness indicators created.")

for name, df in clean_datasets.items():
    flags = [
        col for col in df.columns
        if col.endswith("_missing")
    ]

    if flags:
        print(f"\n{name.upper()}")
        display(
            df[flags].sum()
            .rename("missing_flag_count")
            .to_frame()
        )

Missingness indicators created.

CUSTOMERS


,missing_flag_count
email_missing,242
phone_missing,423
date_of_birth_missing,181
city_missing,121
marketing_opt_in_missing,1191
referral_source_missing,1705



PRODUCTS


,missing_flag_count
sub_category_missing,11



ORDERS


,missing_flag_count
order_date_missing,424



ORDER_ITEMS


,missing_flag_count
unit_price_at_purchase_missing,1469



SUPPORT_TICKETS


,missing_flag_count
satisfaction_score_missing,592


In [21]:
print("CUSTOMERS")
print(
    "Birth dates before 1900:",
    int(
        (customers_clean["date_of_birth"]
         < pd.Timestamp("1900-01-01")).sum()
    )
)
print(
    "Birth dates after snapshot date:",
    int(
        (customers_clean["date_of_birth"]
         > pd.Timestamp("2026-07-01")).sum()
    )
)
print(
    "Signup dates after snapshot date:",
    int(
        (customers_clean["signup_date"]
         > pd.Timestamp("2026-07-01")).sum()
    )
)

print("\nPRODUCTS")
print(
    "Unit cost greater than unit price:",
    int(
        (products_clean["unit_cost"]
         > products_clean["unit_price"]).sum()
    )
)

print("\nORDERS")
print(
    "Negative shipping costs:",
    int((orders_clean["shipping_cost"] < 0).sum())
)
print(
    "Discounts outside 0–25:",
    int(
        (
            (orders_clean["discount_pct"] < 0)
            | (orders_clean["discount_pct"] > 25)
        ).sum()
    )
)

print("\nORDER ITEMS")
print(
    "Zero or negative quantities:",
    int((order_items_clean["quantity"] <= 0).sum())
)
print(
    "Negative purchase prices:",
    int(
        (order_items_clean["unit_price_at_purchase"] < 0).sum()
    )
)

print("\nSUPPORT TICKETS")
print(
    "Negative resolution times:",
    int(
        (support_tickets_clean["resolution_time_hours"] < 0).sum()
    )
)
print(
    "Satisfaction scores outside 1–5:",
    int(
        (
            (support_tickets_clean["satisfaction_score"] < 1)
            | (support_tickets_clean["satisfaction_score"] > 5)
        ).sum()
    )
)

CUSTOMERS
Birth dates before 1900: 8
Birth dates after snapshot date: 7
Signup dates after snapshot date: 0

PRODUCTS
Unit cost greater than unit price: 6

ORDERS
Negative shipping costs: 8
Discounts outside 0–25: 0

ORDER ITEMS
Zero or negative quantities: 18
Negative purchase prices: 0

SUPPORT TICKETS
Negative resolution times: 6
Satisfaction scores outside 1–5: 0


In [22]:
# Step 6.6A: Flag invalid values before correcting them

# 1. Customer birth dates
customers_clean["dob_before_1900_invalid"] = (
    customers_clean["date_of_birth"]
    < pd.Timestamp("1900-01-01")
)

customers_clean["dob_after_snapshot_invalid"] = (
    customers_clean["date_of_birth"]
    > pd.Timestamp("2026-07-01")
)

# Replace implausible birth dates with missing values
invalid_dob = (
    customers_clean["dob_before_1900_invalid"]
    | customers_clean["dob_after_snapshot_invalid"]
)

customers_clean.loc[invalid_dob, "date_of_birth"] = pd.NaT


# 2. Products: flag cost greater than selling price
products_clean["cost_exceeds_price_invalid"] = (
    products_clean["unit_cost"] > products_clean["unit_price"]
)

# Keep both prices unchanged until the inconsistency can be investigated.


# 3. Orders: negative shipping costs
orders_clean["negative_shipping_cost_invalid"] = (
    orders_clean["shipping_cost"] < 0
)

orders_clean.loc[
    orders_clean["negative_shipping_cost_invalid"],
    "shipping_cost"
] = np.nan


# 4. Order items: zero or negative quantities
order_items_clean["nonpositive_quantity_invalid"] = (
    order_items_clean["quantity"] <= 0
)

order_items_clean.loc[
    order_items_clean["nonpositive_quantity_invalid"],
    "quantity"
] = pd.NA


# 5. Support tickets: negative resolution times
support_tickets_clean["negative_resolution_time_invalid"] = (
    support_tickets_clean["resolution_time_hours"] < 0
)

support_tickets_clean.loc[
    support_tickets_clean["negative_resolution_time_invalid"],
    "resolution_time_hours"
] = np.nan


print("Invalid-value flags created and selected invalid values handled.")
print("\nInvalid values identified:")
print(
    "Customer birth dates:",
    int(invalid_dob.sum())
)
print(
    "Products with cost above price:",
    int(products_clean["cost_exceeds_price_invalid"].sum())
)
print(
    "Negative shipping costs:",
    int(orders_clean["negative_shipping_cost_invalid"].sum())
)
print(
    "Nonpositive order quantities:",
    int(order_items_clean["nonpositive_quantity_invalid"].sum())
)
print(
    "Negative resolution times:",
    int(support_tickets_clean["negative_resolution_time_invalid"].sum())
)

Invalid-value flags created and selected invalid values handled.

Invalid values identified:
Customer birth dates: 15
Products with cost above price: 6
Negative shipping costs: 8
Nonpositive order quantities: 18
Negative resolution times: 6


In [23]:
print("CUSTOMERS")
print(
    "Remaining implausible birth dates:",
    int(
        (
            (customers_clean["date_of_birth"] < pd.Timestamp("1900-01-01"))
            | (customers_clean["date_of_birth"] > pd.Timestamp("2026-07-01"))
        ).sum()
    )
)

print("\nORDERS")
print(
    "Remaining negative shipping costs:",
    int((orders_clean["shipping_cost"] < 0).sum())
)

print("\nORDER ITEMS")
print(
    "Remaining nonpositive quantities:",
    int((order_items_clean["quantity"] <= 0).sum())
)

print("\nSUPPORT TICKETS")
print(
    "Remaining negative resolution times:",
    int((support_tickets_clean["resolution_time_hours"] < 0).sum())
)

print("\nRows preserved:")
for name, df in clean_datasets.items():
    print(f"{name}: {len(df):,} rows")

CUSTOMERS
Remaining implausible birth dates: 0

ORDERS
Remaining negative shipping costs: 0

ORDER ITEMS
Remaining nonpositive quantities: 0

SUPPORT TICKETS
Remaining negative resolution times: 0

Rows preserved:
customers: 6,090 rows
products: 220 rows
orders: 28,272 rows
order_items: 73,465 rows
support_tickets: 5,200 rows


In [24]:
# Step 6.7A: Detect numerical outliers using the IQR method

outlier_columns = {
    "products": ["unit_cost", "unit_price"],
    "orders": ["shipping_cost", "discount_pct", "total_amount"],
    "order_items": ["quantity", "unit_price_at_purchase"],
    "support_tickets": [
        "resolution_time_hours",
        "satisfaction_score"
    ]
}

outlier_results = []

for dataset_name, columns in outlier_columns.items():
    df = clean_datasets[dataset_name]

    for col in columns:
        series = pd.to_numeric(df[col], errors="coerce").dropna()

        if len(series) == 0:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (series < lower_bound) |
            (series > upper_bound)
        )
        outlier_results.append({
            "dataset": dataset_name,
            "column": col,
            "Q1": round(q1, 2),
            "Q3": round(q3, 2),
            "IQR": round(iqr, 2),
            "lower_bound": round(lower_bound, 2),
            "upper_bound": round(upper_bound, 2),
            "outlier_count": int(outlier_mask.sum()),
            "outlier_percent": round(
                100 * outlier_mask.sum() / len(series), 2
            )
        })

outlier_report = pd.DataFrame(outlier_results)

display(
    outlier_report.sort_values(
        ["dataset", "outlier_count"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

,dataset,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percent
0,order_items,quantity,1.00,2.00,1.00,-0.50,3.50,3747,5.10
1,order_items,unit_price_at_purchase,174.68,621.12,446.45,-494.99,1290.80,0,0.00
2,orders,discount_pct,0.00,5.00,5.00,-7.50,12.50,3408,12.05
3,orders,total_amount,671.42,2125.37,1453.96,-1509.52,4306.31,862,3.05
4,orders,shipping_cost,0.00,7.99,7.99,-11.98,19.98,0,0.00
5,products,unit_cost,104.13,309.52,205.39,-203.95,617.60,4,1.82
6,products,unit_price,180.96,644.54,463.58,-514.40,1339.91,0,0.00
7,support_tickets,resolution_time_hours,5.20,24.70,19.50,-24.05,53.95,269,5.18
8,support_tickets,satisfaction_score,3.00,5.00,2.00,0.00,8.00,0,0.00


In [25]:
print("LARGEST ORDER QUANTITIES")
display(
    order_items_clean[
        ["order_item_id", "order_id", "product_id", "quantity"]
    ]
    .sort_values("quantity", ascending=False)
    .head(10)
)

print("\nLARGEST ORDER DISCOUNTS")
display(
    orders_clean[
        ["order_id", "discount_pct", "total_amount", "order_status"]
    ]
    .sort_values("discount_pct", ascending=False)
    .head(10)
)

print("\nLARGEST ORDER TOTALS")
display(
    orders_clean[
        ["order_id", "total_amount", "discount_pct", "order_status"]
    ]
    .sort_values("total_amount", ascending=False)
    .head(10)
)

print("\nLONGEST SUPPORT RESOLUTION TIMES")
display(
    support_tickets_clean[
        ["ticket_id", "issue_category", "resolution_time_hours"]
    ]
    .sort_values("resolution_time_hours", ascending=False)
    .head(10)
)

LARGEST ORDER QUANTITIES


,order_item_id,order_id,product_id,quantity
73370,OI515245,ORD122859,PROD1217,4
73352,OI525493,ORD110803,PROD1219,4
73356,OI542966,ORD118405,PROD1174,4
12,OI517523,ORD103742,PROD1071,4
73457,OI542064,ORD102068,PROD1203,4
73459,OI502868,ORD125273,PROD1030,4
73359,OI557555,ORD113344,PROD1109,4
11172,OI522440,ORD117555,PROD1075,4
11177,OI520542,ORD122964,PROD1005,4
73493,OI536853,ORD121403,PROD1144,4



LARGEST ORDER DISCOUNTS


,order_id,discount_pct,total_amount,order_status
17367,ORD112399,25,2208.39,Completed
8202,ORD105632,25,1660.48,Completed
1977,ORD121283,25,780.59,Completed
26103,ORD128177,25,879.54,Cancelled
9386,ORD123378,25,2088.04,Completed
23987,ORD103793,25,171.87,Cancelled
9385,ORD104156,25,828.91,Completed
1468,ORD107300,25,362.51,Completed
1968,ORD111762,25,1113.57,Completed
5686,ORD102699,25,1086.0,Completed



LARGEST ORDER TOTALS


,order_id,total_amount,discount_pct,order_status
2843,ORD122659,10536.77,0,Cancelled
7384,ORD109142,9937.54,0,Completed
18999,ORD104410,9092.24,0,Returned
8441,ORD118578,8905.2,0,Completed
16744,ORD127704,8826.77,0,Completed
14297,ORD115690,8688.53,0,Completed
10957,ORD106103,8651.32,0,Completed
15005,ORD127851,8648.08,0,Completed
801,ORD115492,8417.84,0,Completed
11374,ORD118514,8264.33,0,Completed



LONGEST SUPPORT RESOLUTION TIMES


,ticket_id,issue_category,resolution_time_hours
1422,TIX101422,Billing Issue,782.0
854,TIX100854,Billing Issue,682.0
190,TIX100190,Account Access,542.0
3752,TIX103752,Wrong Item Shipped,374.0
781,TIX100781,Product Question,312.0
1109,TIX101109,Delivery Delay,230.0
4878,TIX104878,Refund Status,222.0
2542,TIX102542,Product Question,139.2
2992,TIX102992,Delivery Delay,134.2
5172,TIX105172,Account Access,125.8


In [26]:
# Step 6.8: Referential integrity checks

customer_ids = set(customers_clean["customer_id"].dropna())
product_ids = set(products_clean["product_id"].dropna())
order_ids = set(orders_clean["order_id"].dropna())

orphan_orders = orders_clean[
    ~orders_clean["customer_id"].isin(customer_ids)
]

orphan_order_items_orders = order_items_clean[
    ~order_items_clean["order_id"].isin(order_ids)
]

orphan_order_items_products = order_items_clean[
    ~order_items_clean["product_id"].isin(product_ids)
]

orphan_tickets = support_tickets_clean[
    ~support_tickets_clean["customer_id"].isin(customer_ids)
]

print("Orders linked to nonexistent customers:", len(orphan_orders))
print("Order items linked to nonexistent orders:", len(orphan_order_items_orders))
print("Order items linked to nonexistent products:", len(orphan_order_items_products))
print("Support tickets linked to nonexistent customers:", len(orphan_tickets))

print("\nExample orphan orders:")
display(orphan_orders.head(10))

Orders linked to nonexistent customers: 20
Order items linked to nonexistent orders: 0
Order items linked to nonexistent products: 0
Support tickets linked to nonexistent customers: 0

Example orphan orders:


,order_id,customer_id,order_date,channel,order_status,payment_method,shipping_cost,discount_pct,total_amount,order_date_missing,negative_shipping_cost_invalid
4248,ORD115601,CUST900013,2025-11-04,Mobile App,Completed,Gift Card,0.0,0,851.62,0,False
5700,ORD127135,CUST900006,2025-07-05,Mobile App,Completed,Store Credit,12.99,0,1326.47,0,False
6224,ORD102970,CUST900004,2026-03-17,In-Store,Completed,Debit Card,5.99,0,516.4,0,False
8497,ORD104582,CUST900014,2026-01-16,Mobile App,Completed,Store Credit,12.99,0,620.02,0,False
10283,ORD109519,CUST900015,2023-05-05,Online,Completed,Credit Card,0.0,0,2462.51,0,False
10551,ORD111134,CUST900003,2024-12-22,In-Store,Completed,Credit Card,7.99,5,193.91,0,False
10966,ORD116460,CUST900001,2022-11-07,In-Store,Returned,Store Credit,0.0,25,250.53,0,False
11127,ORD124916,CUST900005,2025-06-12,In-Store,Completed,Store Credit,4.99,15,1800.01,0,False
11385,ORD105419,CUST900017,2024-08-03,Online,Completed,Debit Card,5.99,10,1024.51,0,False
12112,ORD109372,CUST900011,2024-08-04,In-Store,Completed,Debit Card,12.99,0,545.61,0,False


In [27]:
# Flag orders whose customer IDs are absent from the customer dataset

customer_ids = set(customers_clean["customer_id"].dropna())

orders_clean["customer_reference_missing"] = (
    ~orders_clean["customer_id"].isin(customer_ids)
).astype("int8")

print("Orders with missing customer references:")
print(int(orders_clean["customer_reference_missing"].sum()))

print("\nOrder rows preserved:", len(orders_clean))

display(
    orders_clean.loc[
        orders_clean["customer_reference_missing"] == 1,
        ["order_id", "customer_id", "order_date",
         "order_status", "total_amount"]
    ].head(10)
)

Orders with missing customer references:
20

Order rows preserved: 28272


,order_id,customer_id,order_date,order_status,total_amount
4248,ORD115601,CUST900013,2025-11-04,Completed,851.62
5700,ORD127135,CUST900006,2025-07-05,Completed,1326.47
6224,ORD102970,CUST900004,2026-03-17,Completed,516.4
8497,ORD104582,CUST900014,2026-01-16,Completed,620.02
10283,ORD109519,CUST900015,2023-05-05,Completed,2462.51
10551,ORD111134,CUST900003,2024-12-22,Completed,193.91
10966,ORD116460,CUST900001,2022-11-07,Returned,250.53
11127,ORD124916,CUST900005,2025-06-12,Completed,1800.01
11385,ORD105419,CUST900017,2024-08-03,Completed,1024.51
12112,ORD109372,CUST900011,2024-08-04,Completed,545.61


In [28]:
# Step 6.9: Final quality summary before export

final_quality_summary = []

for name, df in clean_datasets.items():
    final_quality_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum())
    })

final_quality_summary = pd.DataFrame(final_quality_summary)

display(final_quality_summary)

print(
    "\nTotal rows:",
    f"{final_quality_summary['rows'].sum():,}"
)
print(
    "Total missing cells:",
    f"{final_quality_summary['missing_cells'].sum():,}"
)

,dataset,rows,columns,missing_cells,exact_duplicate_rows
0,customers,6090,22,3878,0
1,products,220,11,11,0
2,orders,28272,12,432,0
3,order_items,73465,7,1487,0
4,support_tickets,5200,10,598,0



Total rows: 113,247
Total missing cells: 6,406


In [29]:
# Refresh the dictionary with the latest cleaned datasets

clean_datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "support_tickets": support_tickets_clean
}

# Create the output folder in Google Drive
CLEAN_DATA_PATH = (
    "/content/drive/MyDrive/Meridian_Churn_Capstone/Data/Cleaned"
)

os.makedirs(CLEAN_DATA_PATH, exist_ok=True)

# Export each cleaned dataset
for name, df in clean_datasets.items():
    output_file = os.path.join(CLEAN_DATA_PATH, f"{name}.csv")
    df.to_csv(output_file, index=False)

    print(
        f"Saved {name}.csv: "
        f"{len(df):,} rows, {len(df.columns)} columns"
    )

print("\nExport folder:")
print(CLEAN_DATA_PATH)

Saved customers.csv: 6,090 rows, 22 columns
Saved products.csv: 220 rows, 11 columns
Saved orders.csv: 28,272 rows, 12 columns
Saved order_items.csv: 73,465 rows, 7 columns
Saved support_tickets.csv: 5,200 rows, 10 columns

Export folder:
/content/drive/MyDrive/Meridian_Churn_Capstone/Data/Cleaned


In [30]:
# Reload each exported CSV and compare its dimensions

verification_results = []

for name, original_df in clean_datasets.items():
    file_path = os.path.join(CLEAN_DATA_PATH, f"{name}.csv")

    reloaded_df = pd.read_csv(file_path)

    verification_results.append({
        "dataset": name,
        "original_rows": len(original_df),
        "exported_rows": len(reloaded_df),
        "original_columns": len(original_df.columns),
        "exported_columns": len(reloaded_df.columns),
         "dimensions_match": (
            original_df.shape == reloaded_df.shape
        ),
        "file_exists": os.path.exists(file_path)
    })

verification_report = pd.DataFrame(verification_results)
display(verification_report)

print(
    "\nAll files verified:",
    bool(
        verification_report["file_exists"].all()
        and verification_report["dimensions_match"].all()
    )
)

,dataset,original_rows,exported_rows,original_columns,exported_columns,dimensions_match,file_exists
0,customers,6090,6090,22,22,True,True
1,products,220,220,11,11,True,True
2,orders,28272,28272,12,12,True,True
3,order_items,73465,73465,7,7,True,True
4,support_tickets,5200,5200,10,10,True,True



All files verified: True
